# Customer Feedback Analysis & Automated Apology Email Generator

## Objective

This project analyzes customer reviews from an e-commerce dataset to:

- Clean and preprocess review text.
- Identify critical customer reviews (Ratings ≤ 2).
- Extract common complaint keywords.
- Generate personalized apology emails using Generative AI (Groq Llama 3.3 70B).

Tools Used:

- Python
- Pandas
- Counter
- LangChain
- Groq API

In [90]:
import pandas as pd
from collections import Counter

# Load Dataset

In [104]:
from google.colab import files
uploaded = files.upload()

Saving flipkart_product.csv to flipkart_product (2).csv


#Read the dataset

In [105]:
df = pd.read_csv("flipkart_product.csv")
df.head(189874)

,ProductName,Price,Rate,Review,Summary
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",5,Super!,Great cooler.. excellent air flow and for this...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",5,Awesome,Best budget 2 fit cooler. Nice cooling
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",3,Fair,The quality is good but the power of air is de...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",1,Useless product,Very bad product it's a only a fan
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",3,Fair,Ok ok product
...,...,...,...,...,...
189869,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,5,Terrific,Thanks
189870,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,5,Terrific,Good
189871,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,5,Just wow!,Super
189872,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,4,Worth the money,Good


# shows the column name

In [106]:
print(df.columns)

Index(['ProductName', 'Price', 'Rate', 'Review', 'Summary'], dtype='object')


# Handle Missing Values

In [107]:
#handling null/missing values
print(df.isnull().sum())
df = df.dropna(subset=["Review"]).copy()
df.shape

ProductName     0
Price           1
Rate            1
Review          4
Summary        14
dtype: int64


(189870, 5)

# Clean Review Text

In [108]:
df["Review"] = (
    df["Review"]
    .str.lower()
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)
    .str.strip()
)

# show review column

In [109]:
df["Review"].head(189870)

,Review
0,super
1,awesome
2,fair
3,useless product
4,fair
...,...
189868,terrific
189869,terrific
189870,terrific
189871,just wow


# Filter Critical Reviews

In [110]:
#filter and isolate "Critical" reviews
df["Rate"] = pd.to_numeric(df["Rate"],errors='coerce')
critical_reviews = df[df["Rate"] <= 2]

critical_reviews.head(189870)

,ProductName,Price,Rate,Review,Summary
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",1.0,useless product,Very bad product it's a only a fan
8,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??3,999",1.0,unsatisfactory,Very bad cooler
32,Candes 60 L Room/Personal Air Cooler?ÿ?ÿ(White...,"ƒ??8,999",2.0,expected a better product,Bad quality
48,MAHARAJA WHITELINE 65 L Desert Air Cooler?ÿ?ÿ(...,"ƒ??7,999",1.0,waste of money,Small wire and moter capacity is very low fan ...
59,MAHARAJA WHITELINE 65 L Desert Air Cooler?ÿ?ÿ(...,"ƒ??7,999",1.0,worthless,The product doesn't cool as rated by other cus...
...,...,...,...,...,...
189696,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,1.0,utterly disappointed,This cream is not helpful tor skin
189724,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,1.0,hated it,I would never recommend it for oily skin peopl...
189749,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,2.0,bad quality,Very bad
189834,"NIVEA Soft Light Moisturizer for Face, Hand & ...",â‚¹142,2.0,moderate,Not Bad


# Extract Top Complaint Keywords

In [111]:
# most common complaint keyword
stop_words = {
    "the","is","was","a","an","this","that","and","to",
    "of","for","my","it","i","in","on","with","very",
    "have","had","not"
}

words = []

for review in critical_reviews["Review"]:
    for word in review.split():
        if word not in stop_words:
            words.append(word)

keyword_counts = Counter(words)


## Top 10 Complaint Keywords

In [112]:
top_keywords = keyword_counts.most_common(10)

for word, count in top_keywords:
    print(f"{word}: {count}")

nan: 3824
product: 3269
money: 2378
waste: 2377
disappointed: 2091
better: 1690
absolute: 1265
rubbish: 1265
poor: 1213
utterly: 1212


# Select Top 3 Detailed Critical Reviews

In [113]:
critical_reviews = df[df["Rate"] <= 2].copy()

critical_reviews["Length"] = critical_reviews["Review"].str.len()

top_reviews = critical_reviews.sort_values(
    by="Length",
    ascending=False
).head(3)

In [114]:
!pip install langchain-groq

In [115]:
from langchain_groq import ChatGroq

# Generate AI-Based Apology Emails

In [116]:
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate


llm = ChatGroq(
    groq_api_key="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",
    model="llama-3.3-70b-versatile",

)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an empathetic, professional, and solution-oriented Customer Support Agent. "
        "Your goal is to write a short, highly personalized apology email that directly addresses "
        "the specific pain points and complaints raised by the customer in their review. "
        "Keep the tone sincere, warm, and brief. Use placeholders like [Customer Name] if the name isn't provided."
    ),
    (
        "human",
        "Generate 3 different apology email based on this critical customer review:\n\n{review}"
    )
])


email_generator_chain = prompt | llm


top_3_critical = [critical_reviews.head(3)]


print("--- Generating Apology Emails for Top 3 Critical Reviews ---\n")

for i, review in enumerate(top_3_critical, 1):
    print(f"=== REVIEW #{i} ===")
    print(f"Customer Review: \"{review}\"\n")


    response = email_generator_chain.invoke({"review": review})

    print("Generated Email Response:")
    print(response.content)
    print("\n" + "-"*50 + "\n")

--- Generating Apology Emails for Top 3 Critical Reviews ---

=== REVIEW #1 ===
Customer Review: "                                          ProductName     Price  Rate  \
3   Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...  ƒ??3,999   1.0   
8   Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...  ƒ??3,999   1.0   
32  Candes 60 L Room/Personal Air Cooler?ÿ?ÿ(White...  ƒ??8,999   2.0   

                       Review                             Summary  Length  
3             useless product  Very bad product it's a only a fan      15  
8              unsatisfactory                     Very bad cooler      14  
32  expected a better product                         Bad quality      25  "

Generated Email Response:
Here are three different apology email based on the critical customer review:

**Email 1: For Customer who purchased Candes 12 L Room/Personal Air Cooler**

Subject: A Sincere Apology for Your Candes 12 L Air Cooler Experience

Dear [Customer Name],

I am writing to apologize 